# ST-OMR Meter Background Adaptation v2

Runs the shadow-only real-domain Meter adaptation as a background process. Progress and heartbeat are written atomically to Drive. The monitor shows `phase ?/9`, D10 cache files `?/44260`, `epoch ?/8`, and batch `?/total`. TEST, runtime, Resolver, and production promotion remain closed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_REF = 'fix/meter-real-domain-adaptation-v1'
WORK_ROOT = Path('/content/st-omr-meter-background-v2')
REPO_DIR = WORK_ROOT / 'repo'
TEACHER_BUNDLE = WORK_ROOT / 'teacher-gold-bundle-v1'
D10_LOCAL_CACHE = Path('/content/st-omr-d10-local-cache-v1')
D11_LOCAL_CHECKPOINT = WORK_ROOT / 'd11-checkpoint.pt'
DRIVE_PILOT_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/00_AUDIT/teacher_gold_pilot_v1')
D10_DRIVE_ROOT = Path('/content/drive/MyDrive/ST-OMR-D10/stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a')
D11_DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/ST-OMR-D11-authoritative/c8bc47bab1b8bccad77f42b8b5bdaa499ec0cc96/72132bd250865f350dd88229932b5b00dc9fa01041b0443992b7fa85613cacba/checkpoint-cd2d6192411371628518f4a8327cb0169910425494fa4a82082cd268d85254f3.pt')
D10_MANIFEST_SHA256 = '6927e1bcc5251257a983a306e2f1875c9515f97c6724a8fe9f24382c6ff30db4'
D10_ARTIFACT_BINDING_SHA256 = 'b72e2f5550c727484ea7226561fcd7c8e405d7d83a5bbab199d2780b8bc5db4d'
D11_CHECKPOINT_SHA256 = 'cd2d6192411371628518f4a8327cb0169910425494fa4a82082cd268d85254f3'
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS')
CONTROL_ROOT = DRIVE_RUNS_ROOT / 'meter-background-v2-control'
STATUS_PATH = CONTROL_ROOT / 'status.json'
LOG_PATH = CONTROL_ROOT / 'runner.log'
RUN_ROOT = DRIVE_RUNS_ROOT / 'meter-real-domain-background-v2-run'

def read_status():
    try:
        return json.loads(STATUS_PATH.read_text('ascii'))
    except (FileNotFoundError, OSError, UnicodeError, json.JSONDecodeError):
        return None

def heartbeat_age_seconds(status):
    if not status or not status.get('updated_at'):
        return float('inf')
    stamp = datetime.fromisoformat(status['updated_at'].replace('Z', '+00:00'))
    return (datetime.now(timezone.utc) - stamp).total_seconds()


## Prepare exact code and inputs

A recent heartbeat blocks duplicate launch. A stale/interrupted run may be relaunched; it resumes from the latest complete epoch checkpoint stored in Drive.

In [ ]:
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
CONTROL_ROOT.mkdir(parents=True, exist_ok=True)
prior = read_status()
if prior and prior.get('state') == 'RUNNING' and heartbeat_age_seconds(prior) < 120:
    raise RuntimeError('A background run is already active. Run only the monitor cell below.')
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
repository_sha = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--extra-index-url', 'https://download.pytorch.org/whl/cpu', '-r', str(REPO_DIR / 'requirements-training.txt')], check=True)
pilot_inputs = {
    'pilot': DRIVE_PILOT_ROOT / 'pilot-data.json',
    'choices': DRIVE_PILOT_ROOT / 'ST_OMR_METER_TEACHER_GOLD_PILOT_choices.json',
    'permission': DRIVE_PILOT_ROOT / 'meter-training-permission-evidence-v1.json',
    'privacy': DRIVE_PILOT_ROOT / 'meter-privacy-review-evidence-v1.json',
}
for path in (*pilot_inputs.values(), D10_DRIVE_ROOT / 'COMPLETE', D10_DRIVE_ROOT / 'manifest.json', D10_DRIVE_ROOT / 'receipt.json', D11_DRIVE_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(path)
print({'repository_sha': repository_sha, 'status_path': str(STATUS_PATH), 'run_root': str(RUN_ROOT), 'test_opened': False})


## Start or resume in background

This cell returns immediately. The process continues while the Colab runtime remains alive. Closing the browser does not intentionally stop it, but Colab may reclaim an idle/disconnected runtime; the Drive heartbeat exposes that condition and epoch checkpoints allow relaunch.

In [ ]:
if (RUN_ROOT / 'RUN_COMPLETE').is_file():
    print({'already_complete': True, 'run_root': str(RUN_ROOT)})
    background_pid = None
else:
    prior = read_status()
    if prior and prior.get('state') == 'RUNNING' and heartbeat_age_seconds(prior) < 120:
        print({'already_running': True, 'status': str(STATUS_PATH)})
        background_pid = None
    else:
        command = [
            sys.executable, '-u', str(REPO_DIR / 'tools/meter_real_domain_background_runner_v1.py'),
            '--repository-root', str(REPO_DIR),
            '--pilot', str(pilot_inputs['pilot']), '--choices', str(pilot_inputs['choices']),
            '--permission', str(pilot_inputs['permission']), '--privacy', str(pilot_inputs['privacy']),
            '--teacher-bundle', str(TEACHER_BUNDLE),
            '--d10-drive-root', str(D10_DRIVE_ROOT), '--d10-cache-root', str(D10_LOCAL_CACHE),
            '--d10-manifest-sha256', D10_MANIFEST_SHA256,
            '--d10-artifact-binding-sha256', D10_ARTIFACT_BINDING_SHA256,
            '--d11-drive-checkpoint', str(D11_DRIVE_CHECKPOINT),
            '--d11-local-checkpoint', str(D11_LOCAL_CHECKPOINT), '--d11-sha256', D11_CHECKPOINT_SHA256,
            '--output-root', str(RUN_ROOT), '--status-path', str(STATUS_PATH),
        ]
        log_handle = LOG_PATH.open('a', encoding='utf-8')
        log_handle.write(f'\n=== launch {datetime.now(timezone.utc).isoformat()} repo={repository_sha} ===\n')
        log_handle.flush()
        environment = dict(os.environ)
        environment['PYTHONPATH'] = str(REPO_DIR)
        process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT, env=environment, start_new_session=True)
        background_pid = process.pid
        log_handle.close()
        print({'background_started': True, 'pid': background_pid, 'status': str(STATUS_PATH), 'log': str(LOG_PATH)})


## Live monitor

Stopping this monitor cell does not stop the background runner. Rerun only this cell to reconnect to progress.

In [ ]:
from IPython.display import clear_output
while True:
    status = read_status()
    clear_output(wait=True)
    if not status:
        print('Durum dosyası bekleniyor...')
        time.sleep(3)
        continue
    age = heartbeat_age_seconds(status)
    print(f"DURUM: {status.get('state')} | AŞAMA: {status.get('phase_index', 0)}/{status.get('phase_total', 9)} {status.get('phase')}")
    print(f"EPOCH: {status.get('epoch', status.get('completed_epoch', 0))}/{status.get('epochs_total', 8)} | BATCH: {status.get('batch', 0)}/{status.get('batches_total', 0)}")
    if status.get('files_total'):
        print(f"D10 YEREL KOPYA: {status.get('files_completed', 0)}/{status['files_total']} dosya")
    print(f"SÜRE: {status.get('elapsed_seconds', 0) // 60} dakika | SON HEARTBEAT: {int(age)} saniye önce")
    print(f"OLAY: {status.get('event')}")
    if status.get('result'):
        print(f"SONUÇ: {status['result']}")
    if status.get('error'):
        print(f"HATA: {status.get('error_type')}: {status['error']}")
    if status.get('state') in {'COMPLETE', 'FAILED'}:
        break
    if age > 120:
        print('UYARI: Heartbeat 120 saniyeyi geçti; Colab süreci durmuş veya runtime kopmuş olabilir.')
        break
    time.sleep(5)


## Bounded result

Run after the monitor reaches COMPLETE. If the runner is still active or failed, this cell prints the real status and recent log instead of hiding it behind a generic error. An accepted checkpoint remains shadow-only.

In [ ]:
def show_bounded_result():
    status = read_status()
    if not status:
        print('SONUÇ HAZIR DEĞİL: status.json henüz yok. Monitor hücresini çalıştırın.')
        return None
    if status.get('state') != 'COMPLETE':
        diagnostic = {key: status.get(key) for key in (
            'state', 'phase_index', 'phase_total', 'phase', 'epoch', 'epochs_total',
            'batch', 'batches_total', 'files_completed', 'files_total',
            'event', 'error_type', 'error', 'updated_at',
        )}
        print('SONUÇ HAZIR DEĞİL; gerçek arka plan durumu:')
        print(json.dumps(diagnostic, indent=2, sort_keys=True))
        if LOG_PATH.is_file():
            recent = LOG_PATH.read_text('utf-8', errors='replace').splitlines()[-25:]
            print('\nSON 25 LOG SATIRI:')
            print('\n'.join(recent))
        print('Monitor RUNNING ise bekleyin; FAILED ise yukarıdaki gerçek hatayı kullanın.')
        return None
    metrics_files = sorted(RUN_ROOT.glob('metrics-*.json'))
    if len(metrics_files) != 1:
        raise RuntimeError(f'COMPLETE durumunda tam bir final metrics dosyası bekleniyordu; bulunan={len(metrics_files)}')
    metrics = json.loads(metrics_files[0].read_text('ascii'))
    summary = {
        'status': metrics['status'], 'best_epoch': metrics['best']['epoch'],
        'baseline_real_macro_f1': metrics['baseline']['real_validation']['macro_f1'],
        'best_real_macro_f1': metrics['best']['real_validation']['macro_f1'],
        'best_real_accuracy': metrics['best']['real_validation']['accuracy'],
        'best_synthetic_macro_f1': metrics['best']['synthetic_validation']['macro_f1'],
        'gate': metrics['best']['gate'], 'checkpoint_sha256': metrics['best']['checkpoint_sha256'],
        'test_opened': metrics['test_opened'], 'runtime_connected': metrics['runtime_connected'],
        'production_promotion_authorized': metrics['production_promotion_authorized'],
    }
    print(json.dumps(summary, indent=2, sort_keys=True))
    assert metrics['test_records'] == 0 and metrics['test_opened'] is False
    assert metrics['runtime_connected'] is False and metrics['resolver_connected'] is False
    assert metrics['production_promotion_authorized'] is False
    return summary

bounded_result = show_bounded_result()
